# DeiT III: Revenge of the ViT

**Paper:** H. Touvron, M. Cord, and H. Jégou, "DeiT III: Revenge of the ViT," in *European Conference on Computer Vision (ECCV)*, 2022. [arXiv:2204.07118](https://arxiv.org/abs/2204.07118)

**Summary:** DeiT III demonstrates that plain ViT with the right data augmentation and training recipe (3-Augmentation) outperforms complex self-supervised methods, challenging the notion that architectural complexity drives performance. LayerScale (γ init=1e-6) and mean pooling are the key architectural changes.

## Architecture Overview

```
Input Image [B, 3, H, W]
        │
        ▼  PatchEmbed (Conv2d, patch_size=16 or 14)
Patches [B, N, D]  + learned absolute pos_embed
        │
        ▼  ┌──────────────────────────────────────────────────────────────┐
           │  DeiT3Block × depth                                          │
           │  ┌────────────────────────────────────────────────────────┐  │
           │  │  attn_out = DeiT3Attn(LN(x))  ← Flash Attention       │  │
           │  │  x = x + LayerScale_γ₁(attn_out)   γ init = 1e-6     │  │
           │  │  x = x + LayerScale_γ₂(MLP(LN(x)))                   │  │
           │  └────────────────────────────────────────────────────────┘  │
           └──────────────────────────────────────────────────────────────┘
        │
        ▼  Final LayerNorm
        │
        ▼  Mean Pool over ALL patch tokens  (no CLS token)
        │
        ├──► pooled_output = mean(x, dim=1)   [B, D]
        └──► patch_tokens  = x                [B, N, D]
             feature_maps  = reshape → [B, D, H/p, W/p]
```

### DeiT I → II → III Evolution
| Feature | DeiT I (2020) | DeiT II (not official) | DeiT III (2022) |
|---------|---------------|------------------------|----------------|
| Architecture | Plain ViT | Plain ViT | Plain ViT + LayerScale |
| Pooling | CLS token | CLS token | Mean pool (no CLS) |
| LayerScale | No | No | Yes (γ init=1e-6) |
| Training | DeiT distillation | — | 3-Augmentation (no distillation) |
| Flash Attention | No | No | Yes (PyTorch ≥2.0) |
| Key insight | Distillation token | — | **Recipe > Architecture** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 3, figsize=(17, 6.5))
fig.patch.set_facecolor('#0f1117')

# ── Panel 1: Architecture comparison (CLS vs mean-pool) ──────────────────────
ax1 = axes[0]
ax1.set_facecolor('#0f1117')
ax1.set_xlim(-0.5, 11)
ax1.set_ylim(-1, 13)
ax1.axis('off')
ax1.set_title('CLS Pooling (ViT) vs Mean Pooling (DeiT III)', color='white', fontsize=10, fontweight='bold')

def draw_token(ax, x, y, label, color, h=0.6, w=0.7):
    b = FancyBboxPatch((x-w/2, y-h/2), w, h,
                        boxstyle='round,pad=0.04', facecolor=color,
                        edgecolor='white', linewidth=0.8)
    ax.add_patch(b)
    ax.text(x, y, label, ha='center', va='center', color='white',
            fontsize=7.5, fontweight='bold')

# Left: ViT CLS approach
ax1.text(2.5, 12.5, 'ViT: CLS Pooling', ha='center', color='#4a9eff', fontsize=9, fontweight='bold')
token_labels = ['CLS', 'P₁', 'P₂', 'P₃', 'P₄', 'P₅']
token_colors = ['#c44dff'] + ['#1a4a3a'] * 5
for i, (l, c) in enumerate(zip(token_labels, token_colors)):
    draw_token(ax1, 0.8 + i*0.75, 11.5, l, c)

# Arrows: only CLS has high attention to output
ax1.annotate('', xy=(2.5, 10.45), xytext=(0.8, 11.2),
             arrowprops=dict(arrowstyle='->', color='#c44dff', lw=2))
for i in range(1, 6):
    ax1.plot([0.8 + i*0.75, 2.5], [11.2, 10.5], color='#333', lw=0.7, ls='--')

b_cls = FancyBboxPatch((1.3, 10.0), 2.4, 0.45, boxstyle='round,pad=0.04',
                         facecolor='#c44dff', edgecolor='white', linewidth=1)
ax1.add_patch(b_cls)
ax1.text(2.5, 10.22, 'pooled = x[:, 0]  (CLS)', ha='center', color='white', fontsize=7.5)
ax1.text(2.5, 9.5, 'Asymmetric: CLS learns a\nspecial role different from patches.\nPatch tokens less useful for dense tasks.',
         ha='center', color='#aaa', fontsize=7.5)

# Right: DeiT III mean-pool approach
ax1.text(8.5, 12.5, 'DeiT III: Mean Pool', ha='center', color='#44ff99', fontsize=9, fontweight='bold')
for i, l in enumerate(['P₁', 'P₂', 'P₃', 'P₄', 'P₅']):
    draw_token(ax1, 6.5 + i*0.8, 11.5, l, '#1a4a3a')
for i in range(5):
    ax1.annotate('', xy=(8.5, 10.45), xytext=(6.5 + i*0.8, 11.2),
                 arrowprops=dict(arrowstyle='->', color='#44ff99', lw=1.2))

b_mean = FancyBboxPatch((7.0, 10.0), 3.0, 0.45, boxstyle='round,pad=0.04',
                          facecolor='#1a4a2a', edgecolor='#44ff99', linewidth=1)
ax1.add_patch(b_mean)
ax1.text(8.5, 10.22, 'pooled = mean(x, dim=1)', ha='center', color='white', fontsize=7.5)
ax1.text(8.5, 9.5, 'Symmetric: all patches contribute\nequally. Each patch token is also a\ngood local feature for dense tasks.',
         ha='center', color='#aaa', fontsize=7.5)

# ── Panel 2: LayerScale γ init comparison ────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
ax2.set_title('LayerScale γ Initialisation Effect on Training', color='white', fontsize=10, fontweight='bold')

steps = np.linspace(0, 100, 200)
rng = np.random.default_rng(42)

def simulate_loss(init_val, noise_scale, instability_prob=0.0):
    loss = 5.0
    losses = []
    residual_scale = init_val
    for i, s in enumerate(steps):
        # residual contribution grows as γ grows
        residual_scale = min(residual_scale * 1.02, 1.0)
        base_delta = -0.035 * residual_scale
        noise = rng.normal(0, noise_scale * (1 - s/200))
        if instability_prob > 0 and rng.random() < instability_prob:
            noise += rng.choice([-1, 1]) * rng.uniform(0.5, 1.5)
        loss = max(0.3, loss + base_delta + noise)
        losses.append(loss)
    return np.array(losses)

loss_no_ls   = simulate_loss(1.0,  0.3, instability_prob=0.04)  # no LS, can spike
loss_dinov2  = simulate_loss(1e-5, 0.12, instability_prob=0.005)  # DINOv2 1e-5
loss_deit3   = simulate_loss(1e-6, 0.08, instability_prob=0.001)  # DeiT III 1e-6

ax2.plot(steps, loss_no_ls,  color='#ff4444', linewidth=1.8, label='No LayerScale', alpha=0.85)
ax2.plot(steps, loss_dinov2, color='#ffaa00', linewidth=1.8, label='LayerScale γ=1e-5 (DINOv2)')
ax2.plot(steps, loss_deit3,  color='#44ff99', linewidth=1.8, label='LayerScale γ=1e-6 (DeiT III)')
ax2.set_xlabel('Training step (×100)', color='white')
ax2.set_ylabel('Loss', color='white')
ax2.tick_params(colors='white')
ax2.legend(framealpha=0.3, labelcolor='white', fontsize=8)
for sp in ax2.spines.values():
    sp.set_edgecolor('#444')

ax2.text(50, 4.2,
         'Smaller γ init → more stable early training.\n'
         'DeiT III uses 1e-6 (vs DINOv2\'s 1e-5) because\n'
         '3-Augmentation is more aggressive, causing\n'
         'larger gradient perturbations early on.',
         color='#cccccc', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='#0f1117', edgecolor='#555', alpha=0.9))

# ── Panel 3: 3-Augmentation components ───────────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor('#0f1117')
ax3.set_xlim(0, 10)
ax3.set_ylim(0, 10)
ax3.axis('off')
ax3.set_title('3-Augmentation Training Recipe', color='white', fontsize=10, fontweight='bold')

def info_box(ax, x, y, title, body, title_color='#4a9eff', fc='#1a1a3a'):
    b = FancyBboxPatch((x-3.8, y-0.8), 7.6, 1.6, boxstyle='round,pad=0.1',
                        facecolor=fc, edgecolor=title_color, linewidth=1.2)
    ax.add_patch(b)
    ax.text(x, y+0.35, title, ha='center', va='center', color=title_color,
            fontsize=9, fontweight='bold')
    ax.text(x, y-0.2, body, ha='center', va='center', color='#cccccc', fontsize=7.5)

augmentations = [
    (5, 9.0,  'RandomResizedCrop(224)',
     'Scale: [0.08, 1.0], Ratio: [3/4, 4/3], Bicubic interp', '#4a9eff', '#1a1a3a'),
    (5, 7.0,  'RandAugment(n=2, m=9)',
     '2 randomly chosen augmentations, each with magnitude 9\n(AutoContrast, Equalize, Rotate, Shear, Translate, etc.)', '#ffaa00', '#2a2a1a'),
    (5, 5.0,  'Random Erasing (p=0.25)',
     'Randomly erase a rectangle of scale [0.02, 0.33]\nFills with uniform noise (not mean color)', '#ff6644', '#2a1a1a'),
    (5, 3.0,  'Mixup (α=0.8) + CutMix (α=1.0)',
     'Mixup: interpolate two images and their labels\nCutMix: paste crop from one image into another', '#44ff99', '#1a2a1a'),
    (5, 1.2,  'Label Smoothing (ε=0.1) + Repeated Aug',
     'Smooth targets prevent overconfidence.\nRepeat aug: same image augmented 2× per batch.', '#c44dff', '#2a1a3a'),
]
for x, y, title, body, tc, fc in augmentations:
    info_box(ax3, x, y, title, body, tc, fc)

plt.tight_layout()
plt.savefig('deit3_architecture.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('DeiT III diagram saved.')

## Key Innovations

### 1. LayerScale — γ Initialised at 1e-6 (Smaller than DINOv2)

LayerScale was introduced in CaiT (Touvron et al., 2021). DeiT III reuses it with a **smaller initial value** than DINOv2 (1e-6 vs 1e-5):

$$x \leftarrow x + \underbrace{\gamma_1}_{\in \mathbb{R}^D,\ \gamma \approx 10^{-6}} \odot \text{Attention}(\text{LN}(x))$$

**Why 1e-6 instead of 1e-5?** The 3-Augmentation training recipe creates larger gradient perturbations at the start of training (strong augmentations produce high-variance gradients). A smaller $\gamma$ provides a stronger early constraint, preventing the network from learning harmful patterns in the first few epochs.

At convergence, $\gamma$ may approach 1.0 (full residual) or remain small — the network decides based on the task.

### 2. No CLS Token — Symmetric Mean Pooling

ViT's CLS token creates an **asymmetry**: the CLS token can attend to all patches, but patches are also modified by the CLS token's presence. This means:
- CLS token learns a role different from patch tokens
- Patch tokens encode both local features **and** global context signals (some dimensions are used to signal CLS)

DeiT III removes this asymmetry with mean pooling:
$$z = \frac{1}{N} \sum_{i=1}^{N} \text{LN}(x)_i$$

Benefits:
- All patch tokens are trained equally
- Each patch token is a good local descriptor without needing to encode global summary info
- Simpler: one fewer learnable token
- Better for dense prediction (no CLS takes up attention head capacity)

### 3. The Key Claim: Recipe > Architecture

The paper's core argument is that **the training recipe matters more than architectural complexity**. DeiT III shows:
- Plain ViT-L trained with 3-Augmentation + good hyperparameters
- Achieves **86.5% top-1 on ImageNet-1K** without extra data or self-supervised pre-training
- Outperforms many complex self-supervised methods when fine-tuned

3-Augmentation components (not in backbone code — applied in training pipeline):
1. RandomResizedCrop + horizontal flip
2. RandAugment (n=2, m=9)
3. Random Erasing (p=0.25)
4. Mixup (α=0.8) + CutMix (α=1.0)
5. Label smoothing (ε=0.1) + Repeated augmentation

### 4. Variants
| Name | D | Heads | Layers | Patch | Params |
|------|---|-------|--------|-------|--------|
| DeiT III-S/16 | 384 | 6 | 12 | 16 | ~22M |
| DeiT III-B/16 | 768 | 12 | 12 | 16 | ~86M |
| DeiT III-L/16 | 1024 | 16 | 24 | 16 | ~307M |
| DeiT III-H/14 | 1280 | 16 | 32 | 14 | ~632M |

## Module Walkthrough

### DeiT3Block — Full Block with LayerScale
```python
class DeiT3Block(nn.Module):
    def __init__(self, dim, num_heads, ls_init=1e-6):
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = DeiT3Attention(dim, num_heads)  # Flash Attention
        self.ls1   = LayerScale(dim, init_value=ls_init)  # γ = 1e-6 * ones(D)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = ViTMLP(dim, mlp_ratio=4.0)
        self.ls2   = LayerScale(dim, init_value=ls_init)

    def forward(self, x):
        attn_out, _ = self.attn(self.norm1(x))
        x = x + self.ls1(attn_out)              # scaled residual branch 1
        x = x + self.ls2(self.mlp(self.norm2(x)))  # scaled residual branch 2
        return x
```

### DeiT3Backbone — Forward Pass
```python
def forward(self, x):       # x: [B, 3, H, W]
    x = self.patch_embed(x)              # [B, N, D]
    x = self.pos_drop(x + self.pos_embed)  # add absolute pos embedding
    # No CLS token prepended — all N patches treated equally
    for blk in self.blocks:
        x, _ = blk(x)
    x = self.norm(x)                     # [B, N, D]
    pooled = x.mean(dim=1)               # [B, D] — mean pool over patches
    # Compare: ViT uses x[:, 0] (CLS token)
    feat_map = x.T(1,2).reshape(B, D, G, G)
    return BackboneOutput(pooled, x, [feat_map])
```

### LayerScale — Reused from DINOv2
```python
class LayerScale(nn.Module):
    def __init__(self, dim, init_value=1e-6):  # DeiT III uses 1e-6
        self.gamma = nn.Parameter(init_value * torch.ones(dim))
    def forward(self, x):
        return x * self.gamma   # [B, N, D] * [D] → broadcast over B, N
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
from hale_core.nn.vision import build_vision_backbone

backbone = build_vision_backbone('deit3_small', img_size=224, return_all_attn=True)
backbone.eval()
print(f'Parameters:  {backbone.num_parameters():,}')
print(f'Embed dim:   {backbone.embed_dim}')
print(f'Depth:       {backbone.depth}')

# Inspect initial LayerScale gamma values
block0 = backbone.blocks[0]
print(f'\nBlock 0 - ls1 gamma[:5]: {block0.ls1.gamma[:5].detach().numpy()}')
print(f'Block 0 - ls2 gamma[:5]: {block0.ls2.gamma[:5].detach().numpy()}')
print(f'(All values = 1e-6 at init — near zero residuals)')

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print(f'\npooled_output:  {out.pooled_output.shape}')   # [1, 384]
print(f'patch_tokens:   {out.patch_tokens.shape}')      # [1, 196, 384]
print(f'feature_maps:   {out.feature_maps[0].shape}')   # [1, 384, 14, 14]

# Compare ViT CLS vs DeiT III mean-pool embedding norms
from hale_core.nn.vision import build_vision_backbone
vit = build_vision_backbone('vit_small', img_size=224)
vit.eval()

rng = np.random.default_rng(0)
x_batch = torch.randn(8, 3, 224, 224)
with torch.no_grad():
    out_vit   = vit(x_batch)
    out_deit3 = backbone(x_batch)

vit_pooled_norms   = out_vit.pooled_output.norm(dim=-1).numpy()
deit3_pooled_norms = out_deit3.pooled_output.norm(dim=-1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Pooled Output Norms: ViT (CLS) vs DeiT III (Mean Pool)', color='white', fontsize=12)

for ax, norms, name, color in zip(axes,
                                    [vit_pooled_norms, deit3_pooled_norms],
                                    ['ViT-S (CLS)', 'DeiT III-S (Mean Pool)'],
                                    ['#4a9eff', '#44ff99']):
    ax.set_facecolor('#1a1a2e')
    ax.bar(range(8), norms, color=color, edgecolor='#555')
    ax.set_title(name, color='white')
    ax.set_xlabel('Sample index', color='white')
    ax.set_ylabel('L2 norm', color='white')
    ax.tick_params(colors='white')
    ax.set_ylim(0, max(max(vit_pooled_norms), max(deit3_pooled_norms)) * 1.2)
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig('deit3_pooled_norms.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Random input — both at init, so norms are similar.')
print('With trained weights, DeiT III mean pool tends to produce more uniform norms.')

In [ ]:
# ── Side-by-side model summary table ────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 3.5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')
ax.axis('off')

columns = ['Model', 'Pooling', 'LayerScale γ', 'Flash Attn', 'D', 'Heads', 'Depth', 'Patch', 'Params']
rows = [
    ['ViT-S/16',       'CLS',      '—',     'No',  '384',  '6',  '12', '16', '22M'],
    ['DINOv2-S/14',    'CLS',      '1e-5',  'Yes', '384',  '6',  '12', '14', '22M'],
    ['EVA-02-S/16',    'Mean',     '—',     'No',  '384',  '6',  '12', '16', '22M'],
    ['DeiT III-S/16',  'Mean',     '1e-6',  'Yes', '384',  '6',  '12', '16', '22M'],
    ['DeiT III-H/14',  'Mean',     '1e-6',  'Yes', '1280', '16', '32', '14', '632M'],
]
row_colors = ['#1a1a2e', '#1a2a1a', '#2a1a1a', '#1a2a3a', '#2a1a2a']

table = ax.table(
    cellText=rows,
    colLabels=columns,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2a2a5a')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(row_colors[row - 1])
        cell.set_text_props(color='white')
    cell.set_edgecolor('#444')

ax.set_title('ViT Family Comparison', color='white', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('vit_family_comparison.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## References

### Primary
1. H. Touvron, M. Cord, and H. Jégou, "DeiT III: Revenge of the ViT," *ECCV 2022*. [arXiv:2204.07118](https://arxiv.org/abs/2204.07118)
2. H. Touvron et al., "Going Deeper with Image Transformers," *ICCV 2021*. [arXiv:2103.17239](https://arxiv.org/abs/2103.17239) — CaiT (LayerScale origin)

### Related Works
3. H. Touvron et al., "Training data-efficient image transformers & distillation through attention," *ICML 2021*. [arXiv:2012.12877](https://arxiv.org/abs/2012.12877) — DeiT I (distillation token)
4. A. Dosovitskiy et al., "An Image is Worth 16×16 Words," *ICLR 2021*. [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)
5. M. Oquab et al., "DINOv2: Learning Robust Visual Features without Supervision," *TMLR 2024*. [arXiv:2304.07193](https://arxiv.org/abs/2304.07193)
6. E. D. Cubuk et al., "RandAugment: Practical Automated Data Augmentation," *NeurIPS 2020*. [arXiv:1909.13719](https://arxiv.org/abs/1909.13719)
7. S. Yun et al., "CutMix: Training Strategy that Makes Better Use of Training Batches," *ICCV 2019*. [arXiv:1905.04899](https://arxiv.org/abs/1905.04899)